# 00 – Datenaufbereitung

Dieses Notebook wird **einmalig** ausgeführt und erzeugt die aufbereiteten Datensätze unter `data/processed/`.

**Schritte:**
1. Rohdaten laden und Zeitbereiche prüfen
2. Auf gemeinsamen Zeitraum trimmen
3. Mergen (PV + Wetter + Einstrahlung)
4. Drop irrelevanter Features
5. Check for missing values // TODO merge from ./missing_value_analysis.ipynb
6. Outliner // TODO: merge from ./outlier_analysis.ipynb
7. Vollständigen Feature-Datensatz speichern → `features.csv`


In [1]:
import sys
from pathlib import Path

import pandas as pd

# Projekt-Root ins sys.path aufnehmen (notebooks/ liegt eine Ebene unterhalb)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    load_pv_data,
    load_weather_data,
    load_irradiance_data,
    clip_to_common_range,
)


## 1. Rohdaten laden

In [2]:
pv_raw      = load_pv_data()
weather_raw = load_weather_data()
irr_raw     = load_irradiance_data()

print(f"PV:           {pv_raw['timestamp'].min()}  →  {pv_raw['timestamp'].max()}   ({len(pv_raw):>7,} Zeilen)")
print(f"Wetter:       {weather_raw['timestamp'].min()}  →  {weather_raw['timestamp'].max()}   ({len(weather_raw):>7,} Zeilen)")
print(f"Einstrahlung: {irr_raw['timestamp'].min()}  →  {irr_raw['timestamp'].max()}  ({len(irr_raw):>7,} Zeilen)")

PV:           2022-02-07 14:00:00  →  2025-06-16 23:45:00   (117,636 Zeilen)
Wetter:       2020-01-01 00:00:00  →  2025-10-05 23:00:00   ( 50,520 Zeilen)
Einstrahlung: 2022-02-07 14:00:00+00:00  →  2025-06-16 23:45:00+00:00  (117,640 Zeilen)


## 2. Gemeinsamen Zeitraum bestimmen & trimmen

In [3]:
pv, weather, irr = clip_to_common_range(pv_raw, weather_raw, irr_raw)

common_start = pv["timestamp"].min()
common_end   = pv["timestamp"].max()
print(f"Gemeinsamer Zeitraum: {common_start}  →  {common_end}")
print(f"PV: {len(pv):,} | Wetter: {len(weather):,} | Einstrahlung: {len(irr):,} Zeilen")


Gemeinsamer Zeitraum: 2022-02-07 15:00:00  →  2025-06-16 23:45:00
PV: 117,632 | Wetter: 29,409 | Einstrahlung: 117,632 Zeilen


## 3. Mergen (PV + Wetter + Einstrahlung)

### `merge_features` – Zusammenführen der drei Datenquellen

Die Funktion kombiniert PV- (15-min, lokale naive Zeit), Wetter- (stündlich) und Einstrahlungsdaten (15-min, UTC) zu einem einheitlichen DataFrame:

1. **Basis** ist das 15-min-PV-Raster.
2. **Wetterdaten** liegen stündlich vor und werden per linearer Interpolation auf 15-Minuten-Slots hochskaliert, um Stufensprünge an Stundengrenzen zu vermeiden.
3. **Einstrahlungsdaten** haben UTC-Zeitstempel und werden zuerst in lokale naive Zeit (`Europe/Berlin`) konvertiert.
4. Beide Quellen werden via `merge_asof` (nächster Zeitpunkt, Toleranz ≤ 15 min) an die PV-Basis angefügt.


In [4]:
def merge_features(
    pv_df: pd.DataFrame,
    weather_df: pd.DataFrame,
    irr_df: pd.DataFrame,
    local_tz: str = "Europe/Berlin",
) -> pd.DataFrame:
    base = pv_df.sort_values("timestamp").reset_index(drop=True)

    # Wetterdaten: stündlich → lineare Interpolation auf 15-Minuten-Raster
    # (verhindert Stufensprünge an Stundengrenzen)
    w_cols = ["timestamp", "temperature_2m", "cloud_cover", "cloud_cover_low",
              "relative_humidity_2m"]
    w = (
        weather_df[w_cols]
        .sort_values("timestamp")
        .set_index("timestamp")
        .resample("15min")
        .interpolate(method="linear")
        .reset_index()
    )
    base = pd.merge_asof(base, w, on="timestamp",
                         direction="nearest", tolerance=pd.Timedelta("15min"))

    # Einstrahlungsdaten: UTC-aware → lokale naive Zeit, dann merge_asof
    irr = irr_df[["timestamp", "ghi_cloudy_sky", "ghi_clear_sky"]].copy()
    irr["timestamp"] = (
        irr["timestamp"].dt.tz_convert(local_tz).dt.tz_localize(None)
    )
    irr = irr.sort_values("timestamp").reset_index(drop=True)
    base = pd.merge_asof(base, irr, on="timestamp",
                         direction="nearest", tolerance=pd.Timedelta("15min"))

    return base


In [5]:
df = merge_features(pv, weather, irr)
df["temperature_2m"] = df["temperature_2m"].round(1)
print(f"Nach merge_features(): {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
df.head(3)

Nach merge_features(): 117,632 Zeilen × 22 Spalten


,timestamp,Ladezustand,Batterie (Laden),Batterie (Entladen),Netzeinspeisung,Netzbezug,Solarproduktion Tracker 1,Solarproduktion Tracker 2,Solarproduktion Tracker 3,Solarproduktion,...,Wallbox (ID 0) Netzbezug,Wallbox (ID 0) Solarladeleistung,Wallbox Gesamtladeleistung,Σ Verbrauch,temperature_2m,cloud_cover,cloud_cover_low,relative_humidity_2m,ghi_cloudy_sky,ghi_clear_sky
0,2022-02-07 15:00:00,23,0,0,5,12,72.0,146.0,0.0,218,...,NaN,NaN,NaN,220,4.4,74.0,36.0,62.00,54.51,75.92
1,2022-02-07 15:15:00,23,0,0,2,2,65.0,132.0,0.0,197,...,NaN,NaN,NaN,194,4.1,65.5,32.0,65.25,50.70,70.61
2,2022-02-07 15:30:00,23,0,0,2,2,66.0,135.0,0.0,201,...,NaN,NaN,NaN,198,3.8,57.0,28.0,68.50,46.48,64.74


## 4. Drop irrelevanter features

In [6]:
FEATURE_COLS = [
    "timestamp",
    "Solarproduktion",  # pv_data
    "temperature_2m", "cloud_cover_low",  # weather
    "ghi_cloudy_sky", "ghi_clear_sky",  # irradiance
]

cols_before = df.shape[1]  # number of columns before

df = df[FEATURE_COLS]

print(f"Gedroppt: {cols_before - df.shape[1]} Columns  →  {df.shape[1]:,} Columns verbleiben")


Gedroppt: 16 Columns  →  6 Columns verbleiben


## 5. Check for missing values

In [7]:
# TODO: merge from  ./missing_values_analysis.ipynb

## 6. Outlier detection and handling

In [8]:
## TODO

## 7. Vollständigen Feature-Datensatz speichern

In [9]:
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

features_path = out_dir / "merged.csv"
df.to_csv(features_path, index=False)
print(f"Gespeichert: './data/processed/features.csv'")
print(f"  {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten")
print(f"  Zeitraum: {df['timestamp'].min()} → {df['timestamp'].max()}")

Gespeichert: './data/processed/features.csv'
  117,632 Zeilen × 6 Spalten
  Zeitraum: 2022-02-07 15:00:00 → 2025-06-16 23:45:00
